# Verificando o path dos csvs

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume"))

path,name,size,modificationTime
dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1762908350000
dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1762908357000
dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1762908352000
dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1762908349000
dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1762908352000
dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1762908352000
dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1762908347000
dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1762908346000
dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume/product_category_name_translation.csv,product_category_name_translation.csv,2613,1762908346000


# Importando as bibliotecas

In [0]:
from pyspark.sql.functions import current_timestamp
import requests
from pyspark.sql.functions import current_timestamp
from datetime import datetime

# Criando as duas databases

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS bronze;
CREATE DATABASE IF NOT EXISTS silver;

# Lendo cada csv

In [0]:
# Base path inside your Unity Catalog volume
base_path = "dbfs:/Volumes/atividade2/atividade2_catalogo/atividade2-volume/"

# Mapping of file names to Bronze table names
files_to_tables = {
    "olist_customers_dataset.csv": "bronze.ft_consumidores",
    "olist_geolocation_dataset.csv": "bronze.ft_geolocalizacao",
    "olist_order_items_dataset.csv": "bronze.ft_itens_pedidos",
    "olist_order_payments_dataset.csv": "bronze.ft_pagamentos_pedidos",
    "olist_order_reviews_dataset.csv": "bronze.ft_avaliacoes_pedidos",
    "olist_orders_dataset.csv": "bronze.ft_pedidos",
    "olist_products_dataset.csv": "bronze.ft_produtos",
    "olist_sellers_dataset.csv": "bronze.ft_vendedores",
    "product_category_name_translation.csv": "bronze.dm_categoria_produtos_traducao"
}

In [0]:
for file_name, table_name in files_to_tables.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{base_path}{file_name}")
        .withColumn("ingestion_timestamp", current_timestamp())
    )
    df.write.mode("overwrite").saveAsTable(table_name)
    print(f" Table created: {table_name}")

 Table created: bronze.ft_consumidores
 Table created: bronze.ft_geolocalizacao
 Table created: bronze.ft_itens_pedidos
 Table created: bronze.ft_pagamentos_pedidos
 Table created: bronze.ft_avaliacoes_pedidos
 Table created: bronze.ft_pedidos
 Table created: bronze.ft_produtos
 Table created: bronze.ft_vendedores
 Table created: bronze.dm_categoria_produtos_traducao


# Adicionando a informação sobre o dolar

In [0]:
data_inicio_formatada = "01-01-2016"
data_fim_formatada = "01-01-2020"

# API URL
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(\
dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'\
&$select=dataHoraCotacao,cotacaoCompra&$format=json"

# Request
response = requests.get(url).json()

df_dollar = spark.createDataFrame(response["value"])
df_dollar = df_dollar.withColumn("ingestion_timestamp", current_timestamp())

# Salvando a bronze
df_dollar.write.mode("overwrite").saveAsTable("bronze.dm_cotacao_dolar")

display(df_dollar.limit(10))

cotacaoCompra,dataHoraCotacao,ingestion_timestamp
4.038,2016-01-04 13:12:41.021,2025-11-12T19:36:49.057Z
4.0108,2016-01-05 13:12:41.306,2025-11-12T19:36:49.057Z
4.0297,2016-01-06 13:08:04.506,2025-11-12T19:36:49.057Z
4.0469,2016-01-07 13:07:20.817,2025-11-12T19:36:49.057Z
4.0244,2016-01-08 13:11:21.614,2025-11-12T19:36:49.057Z
4.0147,2016-01-11 13:08:12.021,2025-11-12T19:36:49.057Z
4.0293,2016-01-12 13:10:57.893,2025-11-12T19:36:49.057Z
3.9857,2016-01-13 13:07:21.557,2025-11-12T19:36:49.057Z
4.0217,2016-01-14 13:03:00.94,2025-11-12T19:36:49.057Z
4.0396,2016-01-15 13:08:54.259,2025-11-12T19:36:49.057Z


In [0]:
%sql
SHOW TABLES IN bronze;

database,tableName,isTemporary
bronze,dm_categoria_produtos_traducao,false
bronze,dm_cotacao_dolar,false
bronze,ft_avaliacoes_pedidos,false
bronze,ft_consumidores,false
bronze,ft_geolocalizacao,false
bronze,ft_itens_pedidos,false
bronze,ft_pagamentos_pedidos,false
bronze,ft_pedidos,false
bronze,ft_produtos,false
bronze,ft_vendedores,false
